# 1. Environment, paths, metadata download
(Downloads the Google Sheet into a pandas DataFrame and saves a backup with a timestamp)

In [92]:
import os
import glob
import h5py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# 1. Path to your network NWB files
DATA_DIR = r"\\allen\programs\celltypes\workgroups\hct\SamanthaL\Ephys recordings\Mouse striatum cell types"

# 2. Local directory for saving results
RESULTS_DIR = r"../results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# 3. Google Sheet Metadata Export URL
SHEET_ID = "1CpN3jsWR5c84a57sGSgWtrgvnnLRKjxJgxjQg8Alx1I"
GID = "1820785849"
METADATA_URL = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}"

# 4. Download Google Sheet and Save Timestamped Backup
try:
    metadata_df = pd.read_csv(METADATA_URL)
    print(f"✅ Successfully loaded Google Sheet metadata ({len(metadata_df)} rows).")
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    meta_filename = f"2025_STR_MSN_analysis_{timestamp}.csv"
    
    os.makedirs("../data/metadata", exist_ok=True)
    meta_save_path = os.path.join("../data/metadata", meta_filename)
    metadata_df.to_csv(meta_save_path, index=False)
    print(f"💾 Saved metadata backup to: {meta_save_path}")
    
except Exception as e:
    print(f"❌ Error downloading Google Sheet: {e}")
    metadata_df = pd.DataFrame()

# 5. Find all NWB files
nwb_files = glob.glob(os.path.join(DATA_DIR, "*.nwb"))
print(f"📁 Found {len(nwb_files)} NWB files in network folder.")

✅ Successfully loaded Google Sheet metadata (208 rows).
💾 Saved metadata backup to: ../data/metadata\2025_STR_MSN_analysis_20261001_115414.csv
📁 Found 229 NWB files in network folder.


# 2. Feature Extractor & Universal NWB Loader
(Includes dynamic current-channel searching to fix NaNs in MIES NWB 1.0 files)

In [133]:
#v14 - PRODUCTION EXTRACTOR WITH EXPLICIT COLUMN UNITS & DUAL SAG METRICS
import os
import re
import h5py
import pandas as pd
import numpy as np
from datetime import datetime
from pynwb import NWBHDF5IO

from ipfx.feature_extractor import SpikeFeatureExtractor, SpikeTrainFeatureExtractor


def inspect_nwb_version(file_path):
    try:
        with NWBHDF5IO(file_path, 'r', load_namespaces=True) as io:
            _ = io.read()
            return "2.0"
    except TypeError:
        return "1.0"
    except Exception:
        try:
            with h5py.File(file_path, 'r') as f:
                ver = f.attrs.get("nwb_version", "")
                if hasattr(ver, 'decode'): ver = ver.decode('utf-8')
                if "2." in str(ver) or str(ver).startswith("2"): return "2.0"
        except Exception:
            pass
        return "1.0"


def calc_ap_half_width(t, v, t_thresh, t_peak, v_thresh, v_peak):
    """Calculates AP half-width via exact linear interpolation at V_half in ms."""
    if np.isnan(v_thresh) or np.isnan(v_peak) or v_peak <= v_thresh or np.isnan(t_thresh) or np.isnan(t_peak):
        return np.nan
    
    v_half = v_thresh + 0.5 * (v_peak - v_thresh)
    idx_thresh = np.argmin(np.abs(t - t_thresh))
    idx_peak = np.argmin(np.abs(t - t_peak))
    
    if idx_peak <= idx_thresh:
        return np.nan

    v_rise = v[idx_thresh : idx_peak + 1]
    t_rise = t[idx_thresh : idx_peak + 1]
    above_half_rise = np.where(v_rise >= v_half)[0]
    if len(above_half_rise) == 0: return np.nan
    i_rise = above_half_rise[0]
    if i_rise == 0:
        t_half_rise = t_rise[0]
    else:
        v0, v1 = v_rise[i_rise - 1], v_rise[i_rise]
        t0, t1 = t_rise[i_rise - 1], t_rise[i_rise]
        t_half_rise = t0 + (v_half - v0) * (t1 - t0) / (v1 - v0) if v1 != v0 else t0

    idx_end = min(len(v) - 1, idx_peak + int(0.010 * 25000))
    v_fall = v[idx_peak : idx_end]
    t_fall = t[idx_peak : idx_end]
    below_half_fall = np.where(v_fall <= v_half)[0]
    if len(below_half_fall) == 0: return np.nan
    i_fall = below_half_fall[0]
    if i_fall == 0:
        t_half_fall = t_fall[0]
    else:
        v0, v1 = v_fall[i_fall - 1], v_fall[i_fall]
        t0, t1 = t_fall[i_fall - 1], t_fall[i_fall]
        t_half_fall = t0 + (v_half - v0) * (t1 - t0) / (v1 - v0) if v1 != v0 else t0

    width_ms = (t_half_fall - t_half_rise) * 1000.0
    return width_ms if 0.05 <= width_ms <= 15.0 else np.nan


def calc_adp_v(t, v, t_peak):
    """Calculates After-Depolarization (ADP) voltage rebound post-spike in mV."""
    if np.isnan(t_peak): return np.nan
    idx_peak = np.argmin(np.abs(t - t_peak))
    idx_start = min(len(v) - 1, idx_peak + int(0.002 * 25000))
    idx_end = min(len(v) - 1, idx_peak + int(0.025 * 25000))
    
    if idx_end <= idx_start: return np.nan
    window_v = v[idx_start:idx_end]
    return float(np.max(window_v)) if len(window_v) > 0 else np.nan


def find_main_stimulus_pulse(i, sampling_rate=25000.0, min_duration_ms=100.0):
    """
    Locates the main stimulus pulse window by identifying contiguous blocks 
    longer than min_duration_ms. Ignores brief t=0 ms test pulses and switching noise.
    """
    base_len = min(1000, len(i) // 10)
    i_base = np.mean(i[:base_len])
    
    active_mask = np.abs(i - i_base) > 5.0
    if not np.any(active_mask):
        return None, None, 0.0, i_base
        
    diffs = np.diff(active_mask.astype(int))
    starts = np.where(diffs == 1)[0] + 1
    ends = np.where(diffs == -1)[0] + 1
    
    if active_mask[0]: starts = np.insert(starts, 0, 0)
    if active_mask[-1]: ends = np.append(ends, len(i))
    
    min_samples = int((min_duration_ms / 1000.0) * sampling_rate)
    valid_blocks = [(s, e) for s, e in zip(starts, ends) if (e - s) >= min_samples]
    
    if not valid_blocks:
        return None, None, 0.0, i_base
        
    p_start, p_end = max(valid_blocks, key=lambda b: b[1] - b[0])
    p_len = p_end - p_start
    i_step = np.mean(i[p_start + int(p_len * 0.1) : p_end - int(p_len * 0.1)])
    delta_i = i_step - i_base
    
    return p_start, p_end, delta_i, i_base


def extract_labnotebook_hardware_metrics(file_path):
    """
    Extracts ground-truth hardware metrics from MIES HDF5 labnotebook.
      - bridge_balance(mOhms): MultiClamp Commander setting (Bridge Bal Value)
      - cell_access(mOhms): Voltage-Clamp test pulse access resistance (TP Peak Resistance)
      - holding_current(pA): Active MultiClamp bias current (I-Clamp Holding Level -> TP Baseline pA)
    """
    bridge_bal_val = np.nan
    vc_tp_peak_res = np.nan
    vc_series_res = np.nan
    gen_tp_peak_res = np.nan
    i_clamp_hold = np.nan
    tp_baseline_pa = np.nan

    try:
        with h5py.File(file_path, 'r') as f:
            if 'general/labnotebook' not in f:
                return {
                    "cell_access(mOhms)": np.nan,
                    "bridge_balance(mOhms)": np.nan,
                    "holding_current(pA)": 0.0
                }
            
            ln_group = f['general/labnotebook']
            dev_name = list(ln_group.keys())[0]
            dev_group = ln_group[dev_name]
            
            if 'numericalKeys' in dev_group and 'numericalValues' in dev_group:
                raw_keys = dev_group['numericalKeys'][:]
                keys = [k.decode('utf-8', errors='ignore') if hasattr(k, 'decode') else str(k) 
                        for k in (raw_keys[0] if raw_keys.ndim > 1 else raw_keys)]
                vals = dev_group['numericalValues'][:]
                
                def get_col_idx(key_name):
                    matches = [i for i, k in enumerate(keys) if k.lower() == key_name.lower()]
                    return matches[0] if matches else None

                clamp_mode_idx = get_col_idx('Clamp Mode')
                bridge_bal_idx = get_col_idx('Bridge Bal Value')
                series_res_idx = get_col_idx('Series Resistance')
                tp_peak_idx = get_col_idx('TP Peak Resistance')
                hold_level_idx = get_col_idx('I-Clamp Holding Level')
                tp_baseline_idx = get_col_idx('TP Baseline pA')

                # 1. Bridge Balance Value (Unfiltered by row-level clamp_mode)
                if bridge_bal_idx is not None:
                    p_bb = vals[:, bridge_bal_idx, 0] if vals.ndim == 3 else vals[:, bridge_bal_idx]
                    valid_bb = p_bb[~np.isnan(p_bb) & (p_bb > 0.5) & (p_bb < 150.0)]
                    if len(valid_bb) > 0:
                        bridge_bal_val = np.median(valid_bb)

                # 2. General TP Peak Resistance (Fallback)
                if tp_peak_idx is not None:
                    p_tp = vals[:, tp_peak_idx, 0] if vals.ndim == 3 else vals[:, tp_peak_idx]
                    valid_tp = p_tp[~np.isnan(p_tp) & (p_tp > 0.5) & (p_tp < 150.0)]
                    if len(valid_tp) > 0:
                        gen_tp_peak_res = np.median(valid_tp)

                # 3. Mode-Filtered VC Sweeps (Clamp Mode 0)
                if clamp_mode_idx is not None:
                    clamp_modes = vals[:, clamp_mode_idx, 0] if vals.ndim == 3 else vals[:, clamp_mode_idx]
                    vc_mask = (clamp_modes == 0)

                    if series_res_idx is not None:
                        p_sr = vals[:, series_res_idx, 0] if vals.ndim == 3 else vals[:, series_res_idx]
                        valid_sr = p_sr[vc_mask & ~np.isnan(p_sr) & (p_sr > 0.5) & (p_sr < 150.0)]
                        if len(valid_sr) > 0: vc_series_res = np.median(valid_sr)

                    if tp_peak_idx is not None:
                        p_tp_vc = vals[:, tp_peak_idx, 0] if vals.ndim == 3 else vals[:, tp_peak_idx]
                        valid_tp_vc = p_tp_vc[vc_mask & ~np.isnan(p_tp_vc) & (p_tp_vc > 0.5) & (p_tp_vc < 150.0)]
                        if len(valid_tp_vc) > 0: vc_tp_peak_res = np.median(valid_tp_vc)

                # 4. Holding Current
                if hold_level_idx is not None:
                    p_hold = vals[:, hold_level_idx, 0] if vals.ndim == 3 else vals[:, hold_level_idx]
                    valid_hold = p_hold[~np.isnan(p_hold)]
                    if len(valid_hold) > 0: i_clamp_hold = np.median(valid_hold)

                if tp_baseline_idx is not None:
                    p_base = vals[:, tp_baseline_idx, 0] if vals.ndim == 3 else vals[:, tp_baseline_idx]
                    valid_base = p_base[~np.isnan(p_base)]
                    if len(valid_base) > 0: tp_baseline_pa = np.median(valid_base)

    except Exception:
        pass

    cell_access = np.nan
    if pd.notna(vc_tp_peak_res):
        cell_access = vc_tp_peak_res
    elif pd.notna(vc_series_res):
        cell_access = vc_series_res
    elif pd.notna(gen_tp_peak_res):
        cell_access = gen_tp_peak_res

    bridge_balance = bridge_bal_val if pd.notna(bridge_bal_val) else np.nan

    holding_current = 0.0
    if pd.notna(i_clamp_hold) and abs(i_clamp_hold) > 0.01:
        holding_current = i_clamp_hold
    elif pd.notna(tp_baseline_pa) and abs(tp_baseline_pa) > 0.01:
        holding_current = tp_baseline_pa

    return {
        "cell_access(mOhms)": cell_access,
        "bridge_balance(mOhms)": bridge_balance,
        "holding_current(pA)": holding_current
    }


def extract_all_features_universal(file_path):
    spike_extractor = SpikeFeatureExtractor()
    
    with h5py.File(file_path, 'r') as f:
        acq = f['acquisition/timeseries'] if 'acquisition/timeseries' in f else f['acquisition']
        stim = f['stimulus/presentation'] if 'stimulus/presentation' in f else f.get('stimulus', {})
            
        sweep_keys = sorted([k for k in acq.keys() if k.startswith("data_")],
                            key=lambda x: int(x.split("_")[1]) if len(x.split("_")) > 1 and x.split("_")[1].isdigit() else 0)
        
        baselines, r_in_list, sag_frac_list, sag_ratio_list, tau_list = [], [], [], [], []
        
        lowest_rheo_i = np.inf
        rheo_spikes_df = None
        rheo_train_features = {}
        rheo_stim_amp = np.nan
        rheo_t, rheo_v = None, None
        
        max_spikes_count = 0
        max_firing_rate = 0.0
        supra_stim_amp = np.nan
        supra_spikes_df = None
        supra_train_features = {}
        supra_t, supra_v = None, None
        supra_cv_isi_calc = np.nan
        
        fi_stims, fi_rates = [], []
        
        for skey in sweep_keys:
            if not isinstance(acq[skey], (h5py.Dataset, h5py.Group)): continue
                
            sweep_num = skey.split("_")[1] if len(skey.split("_")) > 1 else "0"
            v_obj = acq[skey]
            v = v_obj['data'][:] if isinstance(v_obj, h5py.Group) and 'data' in v_obj else v_obj[:]
            if np.max(np.abs(v)) < 0.2: v = v * 1000.0  # V -> mV
                
            i = np.zeros_like(v)
            possible_i_keys = [k for k in stim.keys() if f"_{sweep_num}_" in k or k.endswith(f"_{sweep_num}")]
            for k in possible_i_keys:
                cand_obj = stim[k]
                candidate_i = cand_obj['data'][:] if isinstance(cand_obj, h5py.Group) and 'data' in cand_obj else cand_obj[:]
                if np.max(np.abs(candidate_i)) > 0:
                    i = candidate_i
                    break
                    
            if np.max(np.abs(i)) < 1e-3 and np.max(np.abs(i)) > 0: i = i * 1e12  # A -> pA
                
            if isinstance(v_obj, h5py.Group) and 'timestamps' in v_obj:
                t = v_obj['timestamps'][:]
                if np.max(t) > 1000: t = t / 1000.0
            else:
                t = np.arange(len(v)) / 25000.0
                
            # Discard dead / railed traces early
            if np.max(np.abs(v)) > 500.0:
                continue

            # Contiguous step finder
            p_start, p_end, delta_i, i_base = find_main_stimulus_pulse(i)

            if p_start is not None and p_end is not None and (p_end - p_start) > 100:
                p_len = p_end - p_start
                sweep_duration_sec = t[min(p_end, len(t) - 1)] - t[p_start]
                
                v_base = np.mean(v[max(0, p_start - 1000) : max(1, p_start - 50)])
                baselines.append(v_base)
                
                v_steady = np.mean(v[max(p_start, p_end - int(p_len * 0.2)) : p_end])
                v_trough = np.min(v[p_start : p_end])
                delta_v = v_steady - v_base
                
                # --- Subthreshold Passive Features ---
                if delta_i < -5.0:  # Ignore LSB noise (-4.17 pA)
                    r_in = (abs(delta_v) / abs(delta_i)) * 1000.0
                    
                    if 10.0 < r_in < 3500.0:
                        r_in_list.append(r_in)
                    
                        v_deflection = abs(v_trough - v_base)
                        if v_deflection > 0.5:
                            sag_frac = abs(v_trough - v_steady) / v_deflection
                            if 0.0 <= sag_frac <= 1.0:
                                sag_frac_list.append(sag_frac)
                                sag_ratio_list.append(1.0 - sag_frac)
                            
                        v_target = v_base + 0.632 * delta_v
                        tau_idx = np.where(v[p_start:p_end] <= v_target if delta_v < 0 else v[p_start:p_end] >= v_target)[0]
                        if len(tau_idx) > 0:
                            tau_ms = (t[p_start + tau_idx[0]] - t[p_start]) * 1000.0
                            if 0.5 < tau_ms < 200.0:
                                tau_list.append(tau_ms)

                # --- Active Spiking Features ---
                if np.max(v) > -10.0 and delta_i > 5.0:
                    try:
                        spikes = spike_extractor.process(t, v, i)
                        num_spikes = len(spikes)
                        firing_rate = num_spikes / sweep_duration_sec if sweep_duration_sec > 0 else 0
                        fi_stims.append(delta_i)
                        fi_rates.append(firing_rate)
                        
                        train_extractor = SpikeTrainFeatureExtractor(start=t[p_start], end=t[min(p_end, len(t) - 1)])
                        train_feats = train_extractor.process(t, v, i, spikes) if num_spikes > 0 else {}
                        
                        if num_spikes > 0 and delta_i < lowest_rheo_i:
                            lowest_rheo_i = delta_i
                            rheo_stim_amp = delta_i
                            rheo_spikes_df = spikes
                            rheo_train_features = train_feats
                            rheo_t, rheo_v = t, v
                            
                        if num_spikes > max_spikes_count:
                            max_spikes_count = num_spikes
                            max_firing_rate = firing_rate
                            supra_stim_amp = delta_i
                            supra_spikes_df = spikes
                            supra_train_features = train_feats
                            supra_t, supra_v = t, v
                            
                            if num_spikes >= 3 and 'threshold_t' in spikes.columns:
                                isis = np.diff(spikes['threshold_t'].values)
                                supra_cv_isi_calc = np.std(isis) / np.mean(isis) if np.mean(isis) > 0 else np.nan
                    except Exception:
                        pass
            else:
                baselines.append(np.mean(v[:base_len]))

        # --- Aggregations ---
        v_baseline_final = np.mean(baselines) if baselines else np.nan
        r_in_final = np.median(r_in_list) if r_in_list else np.nan
        tau_final = np.median(tau_list) if tau_list else np.nan
        capacitance_final = (tau_final * 1000.0 / r_in_final) if pd.notna(tau_final) and pd.notna(r_in_final) and r_in_final > 0 else np.nan

        fi_slope = np.polyfit(fi_stims, fi_rates, 1)[0] if len(fi_stims) > 1 else np.nan
        
        first_spk = rheo_spikes_df.iloc[0] if rheo_spikes_df is not None and not rheo_spikes_df.empty else {}
        last_spk = rheo_spikes_df.iloc[-1] if rheo_spikes_df is not None and len(rheo_spikes_df) > 1 else first_spk
        
        rheo_ap_width = calc_ap_half_width(
            rheo_t, rheo_v,
            first_spk.get('threshold_t', np.nan), first_spk.get('peak_t', np.nan),
            first_spk.get('threshold_v', np.nan), first_spk.get('peak_v', np.nan)
        ) if rheo_t is not None else np.nan

        rheo_adp_v = first_spk.get('adp_v', np.nan)
        if pd.isna(rheo_adp_v) and rheo_t is not None:
            rheo_adp_v = calc_adp_v(rheo_t, rheo_v, first_spk.get('peak_t', np.nan))

        supra_first_spk = supra_spikes_df.iloc[0] if supra_spikes_df is not None and not supra_spikes_df.empty else {}
        
        supra_ap_width = calc_ap_half_width(
            supra_t, supra_v,
            supra_first_spk.get('threshold_t', np.nan), supra_first_spk.get('peak_t', np.nan),
            supra_first_spk.get('threshold_v', np.nan), supra_first_spk.get('peak_v', np.nan)
        ) if supra_t is not None else np.nan

        final_cv_isi = supra_train_features.get('cv_isi', np.nan)
        if pd.isna(final_cv_isi):
            final_cv_isi = supra_cv_isi_calc

        return {
            "v_rest(mV)": v_baseline_final,
            
            "input_resistance(mOhms)": r_in_final,
            "tau(ms)": tau_final,
            "capacitance(pF)": capacitance_final,
            "sag_fraction": np.median(sag_frac_list) if sag_frac_list else np.nan,
            "sag_ratio": np.median(sag_ratio_list) if sag_ratio_list else np.nan,
            
            "f_i_curve_slope(Hz/pA)": fi_slope,
            "max_firing_rate(Hz)": max_firing_rate,
            
            "rheobase_sweep_stim_amp(pA)": rheo_stim_amp if lowest_rheo_i != np.inf else np.nan,
            "rheobase_sweep_spike_count": len(rheo_spikes_df) if rheo_spikes_df is not None else 0,
            "rheobase_sweep_threshold_t(s)": first_spk.get('threshold_t', np.nan),
            "rheobase_sweep_peak_t(s)": first_spk.get('peak_t', np.nan),
            "rheobase_sweep_threshold_v(mV)": first_spk.get('threshold_v', np.nan),
            "rheobase_sweep_threshold_i(pA)": first_spk.get('threshold_i', np.nan),
            "rheobase_sweep_peak_v(mV)": first_spk.get('peak_v', np.nan),
            "rheobase_sweep_peak_i(pA)": first_spk.get('peak_i', np.nan),
            "rheobase_sweep_ap_amplitude(mV)": (first_spk.get('peak_v', np.nan) - first_spk.get('threshold_v', np.nan)) if pd.notna(first_spk.get('peak_v')) else np.nan,
            "rheobase_sweep_width(ms)": rheo_ap_width,
            "rheobase_sweep_upstroke(V/s)": first_spk.get('upstroke', np.nan),
            "rheobase_sweep_downstroke(V/s)": first_spk.get('downstroke', np.nan),
            "rheobase_sweep_upstroke_downstroke_ratio": first_spk.get('upstroke_downstroke_ratio', np.nan),
            "rheobase_sweep_trough_v(mV)": first_spk.get('trough_v', np.nan),
            "rheobase_sweep_fast_trough_v(mV)": first_spk.get('fast_trough_v', np.nan),
            "rheobase_sweep_adp_v(mV)": rheo_adp_v,
            "rheobase_sweep_latency(ms)": rheo_train_features.get('latency', np.nan) * 1000.0 if pd.notna(rheo_train_features.get('latency')) else np.nan,
            "rheobase_sweep_adaptation_index": rheo_train_features.get('adapt', np.nan),
            "rheobase_sweep_mean_isi(ms)": rheo_train_features.get('mean_isi', np.nan) * 1000.0 if pd.notna(rheo_train_features.get('mean_isi')) else np.nan,
            "rheobase_sweep_first_isi(ms)": rheo_train_features.get('first_isi', np.nan) * 1000.0 if pd.notna(rheo_train_features.get('first_isi')) else np.nan,
            "rheobase_sweep_avg_rate(Hz)": rheo_train_features.get('avg_rate', np.nan),
            
            "ap_width_ratio": (last_spk.get('width', np.nan) / first_spk.get('width', np.nan)) if pd.notna(first_spk.get('width')) and first_spk.get('width') > 0 else np.nan,
            "ap_height_ratio": ((last_spk.get('peak_v', np.nan) - last_spk.get('threshold_v', np.nan)) / (first_spk.get('peak_v', np.nan) - first_spk.get('threshold_v', np.nan))) if pd.notna(first_spk.get('peak_v')) else np.nan,
            "ap_upstroke_ratio": (last_spk.get('upstroke', np.nan) / first_spk.get('upstroke', np.nan)) if pd.notna(first_spk.get('upstroke')) and first_spk.get('upstroke') > 0 else np.nan,
            
            "suprathreshold_sweep_stim_amp(pA)": supra_stim_amp,
            "suprathreshold_sweep_spike_count": max_spikes_count,
            "suprathreshold_sweep_threshold_v(mV)": supra_first_spk.get('threshold_v', np.nan),
            "suprathreshold_sweep_peak_v(mV)": supra_first_spk.get('peak_v', np.nan),
            "suprathreshold_sweep_width(ms)": supra_ap_width,
            "suprathreshold_sweep_cv_isi": final_cv_isi,
            "suprathreshold_sweep_adaptation_index": supra_train_features.get('adapt', np.nan)
        }


def process_file_universal(file_path, meta_df):
    cell_basename = os.path.basename(file_path).lower()
    cell_name = cell_basename
    
    if 'file_path' in meta_df.columns and 'cell_name' in meta_df.columns:
        for _, row in meta_df.iterrows():
            if pd.notna(row['file_path']) and os.path.basename(str(row['file_path'])).lower() == cell_basename:
                cell_name = str(row['cell_name']).strip()
                break
                
    feats = extract_all_features_universal(file_path)
    feats["nwb_version"] = inspect_nwb_version(file_path)
    feats["cell_name"] = cell_name
    feats["file_path"] = file_path
    
    # Ground-truth Hardware Metrics
    hw_metrics = extract_labnotebook_hardware_metrics(file_path)
    feats["holding_current(pA)"] = hw_metrics["holding_current(pA)"]
    feats["bridge_balance(mOhms)"] = hw_metrics["bridge_balance(mOhms)"]
    feats["cell_access(mOhms)"] = hw_metrics["cell_access(mOhms)"]
    
    return feats

# 3. Single Cell Feature Test

In [135]:
import pandas as pd
import numpy as np

# Select a mix of test files (preferring known active/spiking cells if available)
test_files = [f for f in nwb_files if "850495" in f or "880646" in f][:3]
if not test_files:
    test_files = nwb_files[:3]

test_rows = []
for fpath in test_files:
    feats = process_file_universal(fpath, metadata_df if 'metadata_df' in globals() else pd.DataFrame())
    test_rows.append(feats)

df_test_check = pd.DataFrame(test_rows)

print("=" * 100)
print("📊 #v14 PRE-BATCH VERIFICATION SUMMARY")
print("=" * 100)

# Display essential passive, active, and hardware columns with updated #v14 unit headers
display_cols = [
    "cell_name", 
    "v_rest(mV)", 
    "holding_current(pA)", 
    "cell_access(mOhms)", 
    "bridge_balance(mOhms)", 
    "input_resistance(mOhms)", 
    "tau(ms)", 
    "capacitance(pF)", 
    "sag_fraction", 
    "sag_ratio", 
    "rheobase_sweep_stim_amp(pA)",
    "rheobase_sweep_width(ms)", 
    "rheobase_sweep_adp_v(mV)", 
    "suprathreshold_sweep_cv_isi",
    "max_firing_rate(Hz)"
]

# Render only existing columns in the returned dataframe
valid_display_cols = [c for c in display_cols if c in df_test_check.columns]

pd.set_option('display.max_columns', 20)
display(df_test_check[valid_display_cols])

📊 #v14 PRE-BATCH VERIFICATION SUMMARY


,cell_name,v_rest(mV),holding_current(pA),cell_access(mOhms),bridge_balance(mOhms),input_resistance(mOhms),tau(ms),capacitance(pF),sag_fraction,sag_ratio,rheobase_sweep_stim_amp(pA),rheobase_sweep_width(ms),rheobase_sweep_adp_v(mV),suprathreshold_sweep_cv_isi,max_firing_rate(Hz)
0,Ai14-Het-880646.01.09.02,-57.132069,-10.331873,68.619347,14.407814,1244.686401,50.40,40.492127,0.150136,0.849864,15.0,2.830110,-5.5000,NaN,0.5
1,Ai14-Het-880646.01.09.04,-70.011971,10.144020,9.018727,12.393162,77.660233,23.72,305.433028,0.020961,0.979039,140.0,2.077392,-2.4375,0.093509,3.0
2,Ai14-Het-880646.01.09.05,-68.276901,1.815905,61.787107,14.713065,463.115295,30.54,65.944702,0.021928,0.978072,40.0,3.839500,-4.8750,0.369207,11.0


# 4. Batch Extractor for All Network Files

Batch Execution, Metadata Merge, Column Dropping, and Date Sorting

It filters nwb_calc_cols safely using [c for c in ... if c in df_merged.columns] so it will never crash even if a column name varies:

In [136]:
#v2 USE THIS ONE
import os
import pandas as pd
from datetime import datetime

# Define timestamped output destination
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = os.path.join(RESULTS_DIR, f"SSL_M_ephys_features_{timestamp}.csv")

extracted_rows = []
print(f"🚀 Batch Extracting Features for {len(nwb_files)} NWB files...")

# 1. Batch extraction loop
for idx, nwb_path in enumerate(nwb_files, start=1):
    try:
        features = process_file_universal(nwb_path, metadata_df)
        extracted_rows.append(features)
        print(f"[{idx:04d}/{len(nwb_files):04d}] ✅ Processed: {features['cell_name']}")
        
        # Periodic autosave every 50 files
        if idx % 50 == 0 or idx == len(nwb_files):
            pd.DataFrame(extracted_rows).to_csv(out_csv, index=False)
            
    except Exception as e:
        print(f"[{idx:04d}/{len(nwb_files):04d}] ❌ Error on {os.path.basename(nwb_path)}: {e}")

df_ephys = pd.DataFrame(extracted_rows)

# 2. Clean metadata pre-merge to prevent column collisions
meta_cols_to_drop_pre_merge = [
    'v_rest(mV)', 'holding_current(pA)', 'cell_access(mOhms)', 
    'bridge_balance(mOhms)', 'file_name', 'file_path'
]
if not metadata_df.empty:
    meta_clean = metadata_df.drop_duplicates(subset=['cell_name']).copy()
    meta_clean.drop(columns=[c for c in meta_cols_to_drop_pre_merge if c in meta_clean.columns], inplace=True)
    df_merged = pd.merge(df_ephys, meta_clean, on='cell_name', how='left')
else:
    df_merged = df_ephys

# 3. Drop unwanted metadata columns and file_name
cols_to_drop = ['physiology', 'morphology_alexa_fill', 'ROI', 'hemisphere_location', 'tube_id', '20Xlims_link', 'file_name']
df_merged.drop(columns=[c for c in cols_to_drop if c in df_merged.columns], inplace=True)

# 4. Standardize dates to strict YYMMDD format and sort chronologically
if 'date' in df_merged.columns:
    date_str = df_merged['date'].astype(str).str.split('.').str[0].str.zfill(6)
    parsed_dates = pd.to_datetime(date_str, format='%y%m%d', errors='coerce')
    
    # Fallback for standard date string formats if present
    mask_na = parsed_dates.isna()
    if mask_na.any():
        parsed_dates[mask_na] = pd.to_datetime(df_merged.loc[mask_na, 'date'], errors='coerce')
        
    df_merged['date_dt'] = parsed_dates
    df_merged.sort_values(by='date_dt', ascending=True, inplace=True)
    
    df_merged['date'] = df_merged['date_dt'].dt.strftime('%y%m%d')
    df_merged.drop(columns=['date_dt'], inplace=True)

# 5. Enforce target column ordering
target_first_cols = ['date', 'cell_name', 'file_path', 'virus', 'cell_type_targeted', 'reclassified_cell_type', 'fluorescent_label']
target_calc_cols = ['v_rest(mV)', 'holding_current(pA)', 'bridge_balance(mOhms)', 'cell_access(mOhms)']
target_middle_cols = ['nwb_version']

first_cols = [c for c in target_first_cols if c in df_merged.columns]
nwb_calc_cols = [c for c in target_calc_cols if c in df_merged.columns]
middle_cols = [c for c in target_middle_cols if c in df_merged.columns]

ephys_cols = [c for c in df_merged.columns if c not in first_cols + nwb_calc_cols + middle_cols]

final_column_order = first_cols + nwb_calc_cols + middle_cols + ephys_cols
df_final = df_merged[final_column_order]

# 6. Save final output CSV
df_final.to_csv(out_csv, index=False)

print("\n" + "=" * 70)
print(f"🎉 PRODUCTION EXTRACTION & MERGE COMPLETE!")
print("=" * 70)
print(f"• Total Cells Processed : {len(df_final)}")
print(f"• Total Columns Output  : {df_final.shape[1]}")
print(f"• Output File Path      : {out_csv}")

display(df_final.head(10))

🚀 Batch Extracting Features for 229 NWB files...
[0001/0229] ✅ Processed: C57BL6J-803061.01.01
[0002/0229] ✅ Processed: C57BL6J-803061.01.02
[0003/0229] ✅ Processed: C57BL6J-795017.01.01
[0004/0229] ✅ Processed: C57BL6J-795017.01.02
[0005/0229] ✅ Processed: C57BL6J-795017.01.03
[0006/0229] ✅ Processed: ai14-het-817117.02.02.01_itc18usb_dev_0-compressed.nwb
[0007/0229] ✅ Processed: Ai14-Het-817118.01.01.01
[0008/0229] ✅ Processed: Ai14-Het-817118.02.01.02
[0009/0229] ✅ Processed: Ai14-Het-826949.01.09.01
[0010/0229] ✅ Processed: Ai14-Het-826949.01.09.02
[0011/0229] ✅ Processed: Ai14-Het-826949.01.09.03
[0012/0229] ✅ Processed: Ai14-Het-826949.01.09.04
[0013/0229] ✅ Processed: Ai14-Het-826950.01.01.02
[0014/0229] ✅ Processed: ai14-het-826950.02.09.02_itc18usb_dev_0-compressed.nwb
[0015/0229] ✅ Processed: Ai14-Het-826951.01.09.01
[0016/0229] ✅ Processed: Ai14-Het-826951.02.09.02
[0017/0229] ✅ Processed: Ai14-Het-826951.02.09.03
[0018/0229] ✅ Processed: Ai14-Het-826951.02.09.04
[0019/0229]

c:\mcpython3\envs\Aipfx\lib\site-packages\pynwb\core.py:52: UserWarning: Use of icephys_filtering is deprecated and will be removed in PyNWB 4.0. Use the IntracellularElectrode.filtering field instead
  warn(error_msg)


[0066/0229] ✅ Processed: Ai14-Het-880646.03.03.01


c:\mcpython3\envs\Aipfx\lib\site-packages\pynwb\core.py:52: UserWarning: Use of icephys_filtering is deprecated and will be removed in PyNWB 4.0. Use the IntracellularElectrode.filtering field instead
  warn(error_msg)


[0067/0229] ✅ Processed: Ai14-Het-880646.03.03.02


...

c:\mcpython3\envs\Aipfx\lib\site-packages\pynwb\core.py:52: UserWarning: Use of icephys_filtering is deprecated and will be removed in PyNWB 4.0. Use the IntracellularElectrode.filtering field instead
  warn(error_msg)


[0092/0229] ✅ Processed: Ai14-Het-884383.03.09.02


c:\mcpython3\envs\Aipfx\lib\site-packages\pynwb\core.py:52: UserWarning: Use of icephys_filtering is deprecated and will be removed in PyNWB 4.0. Use the IntracellularElectrode.filtering field instead
  warn(error_msg)


[0093/0229] ✅ Processed: Ai14-Het-884383.03.09.03
[0094/0229] ✅ Processed: Ai14-Het-884744.01.09.01
[0095/0229] ✅ Processed: Ai14-Het-884744.01.09.03
[0096/0229] ✅ Processed: Ai14-Het-884744.02.09.01
[0097/0229] ✅ Processed: c57bl6j-801218.01_icj_2025_07_25_100210-compressed.nwb
[0098/0229] ✅ Processed: c57bl6j-801218.01_icj_2025_07_25_100210-compressedv2.nwb
[0099/0229] ✅ Processed: C57BL6J-801218.02
[0100/0229] ✅ Processed: C57BL6J-802013.02_cholinergic
[0101/0229] ✅ Processed: C57BL6J-802013.03_cholinergic
[0102/0229] ✅ Processed: C57BL6J-803059.01_MSN
[0103/0229] ✅ Processed: C57BL6J-803059.02_ICJ
[0104/0229] ✅ Processed: C57BL6J-803059.03_MSN
[0105/0229] ✅ Processed: C57BL6J-803060.01_MSN
[0106/0229] ✅ Processed: c57bl6j-803060.02_icj_itc18usb_dev_0-compressed.nwb
[0107/0229] ✅ Processed: C57BL6J-803060.03_MSN
[0108/0229] ✅ Processed: C57BL6J-803060.04_cholinergic
[0109/0229] ✅ Processed: C57BL6J-803064.01_cholinergic
[0110/0229] ✅ Processed: C57BL6J-812790.01​.01
[0111/0229] ✅ Pr

,date,cell_name,file_path,virus,cell_type_targeted,reclassified_cell_type,fluorescent_label,v_rest(mV),holding_current(pA),bridge_balance(mOhms),...,ap_width_ratio,ap_height_ratio,ap_upstroke_ratio,suprathreshold_sweep_stim_amp(pA),suprathreshold_sweep_spike_count,suprathreshold_sweep_threshold_v(mV),suprathreshold_sweep_peak_v(mV),suprathreshold_sweep_width(ms),suprathreshold_sweep_cv_isi,suprathreshold_sweep_adaptation_index
0,250527,C57BL6J-803061.01.01,\\allen\programs\celltypes\workgroups\hct\Sama...,no,Str MSN,NaN,no,-105.903496,54.077305,NaN,...,1.000000,1.032806,1.014103,267.201263,10,-23.34375,57.87500,1.325004,0.168576,0.033809
1,250527,C57BL6J-803061.01.02,\\allen\programs\celltypes\workgroups\hct\Sama...,no,Str MSN,NaN,no,-37.379532,121.083229,10.683761,...,1.000000,0.989376,0.951582,267.201263,8,-35.25000,54.84375,1.616549,0.139260,0.031710
2,250603,C57BL6J-795017.01.01,\\allen\programs\celltypes\workgroups\hct\Sama...,AiP14609,Str MSN,NaN,no,-74.566963,49.956360,12.332112,...,1.000000,1.000000,1.000000,267.201263,43,-42.71875,43.53125,1.654347,0.191851,0.011164
3,250603,C57BL6J-795017.01.02,\\allen\programs\celltypes\workgroups\hct\Sama...,AiP14609,Str MSN,NaN,no,-42.877041,-46.165055,13.919414,...,1.048387,0.984689,0.870137,267.201263,40,-33.03125,36.00000,2.304707,0.150700,0.009160
4,250603,C57BL6J-795017.01.03,\\allen\programs\celltypes\workgroups\hct\Sama...,AiP14609,Str MSN,NaN,no,-61.897320,-201.988777,8.485958,...,1.068182,0.917605,0.778528,267.201263,27,-41.87500,56.15625,1.663341,0.229519,0.018548
101,250710,C57BL6J-803059.01_MSN,\\allen\programs\celltypes\workgroups\hct\Sama...,no,Str MSN,NaN,no,-68.540176,-25.208074,9.676434,...,1.081633,0.958738,0.844224,267.201263,24,-29.28125,55.37500,1.727766,0.109877,0.008153
102,250710,C57BL6J-803059.02_ICJ,\\allen\programs\celltypes\workgroups\hct\Sama...,no,OT ICJ,NaN,no,-42.284077,-27.489042,9.218559,...,1.054054,1.081690,0.864718,48.222363,39,-29.18750,-8.68750,1.682894,0.190724,0.000761
103,250710,C57BL6J-803059.03_MSN,\\allen\programs\celltypes\workgroups\hct\Sama...,no,Str MSN,ICJ,no,-50.241631,-9.103024,7.997558,...,1.280702,0.732548,0.563533,102.967087,19,-15.68750,23.03125,1.583392,0.605927,0.041506
100,250723,C57BL6J-802013.03_cholinergic,\\allen\programs\celltypes\workgroups\hct\Sama...,AiP15385,Str CHAT,NaN,yes,-59.785713,-200.000003,9.035409,...,1.000000,1.000000,1.000000,52.035049,11,-39.78125,32.87500,1.520456,0.719840,0.104093
99,250723,C57BL6J-802013.02_cholinergic,\\allen\programs\celltypes\workgroups\hct\Sama...,AiP15385,Str CHAT,NaN,yes,-62.029297,-100.000001,7.753358,...,1.183099,0.984280,0.895370,55.933182,13,-45.59375,36.65625,1.495329,0.420802,0.070431
